In [ ]:
%reload_ext autoreload

# Modell Tunining von Gradientboosting auf unseren Daten

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from scipy.stats import loguniform, randint
from utils import bewerten, trainieren_und_bewerten

---
## Daten laden

In [ ]:
from pathlib import Path

ordner = Path('../data/processed')

X_train_raw = pd.read_parquet(ordner / 'X_train_raw.parquet')
X_test_raw = pd.read_parquet(ordner / 'X_test_raw.parquet')
y_train = pd.read_parquet(ordner / 'y_train.parquet')['loan_status']
y_test = pd.read_parquet(ordner / 'y_test.parquet')['loan_status']

print(X_train_raw.shape, X_test_raw.shape, y_train.shape, y_test.shape)

---
## Tuning beginnen

### Ungetuned

In [ ]:
model_gb = HistGradientBoostingClassifier(random_state=42)
model_gb.fit(X_train_raw, y_train)
bewerten('Gradient Boosting (Standard)', y_test, model_gb.predict(X_test_raw))

---
### Schritt 1

In [ ]:
param_dist = {
    'learning_rate': loguniform(0.01, 0.3),
    'max_iter': randint(100, 1000),
    'max_leaf_nodes': randint(15, 64),
    'min_samples_leaf': randint(10, 100),
    'l2_regularization': loguniform(1e-3, 10),
}

suche = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=40,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
suche.fit(X_train_raw, y_train)

print('Beste Parameter:', suche.best_params_)
print(f'Fehlerquote Cross-Validation: {1 - suche.best_score_:.1%}')

---
### Anwendung der gefundenen Parameter auf die Testdaten

In [ ]:
model_gb_tuned = suche.best_estimator_
bewerten('Gradient Boosting (getunt)', y_test, model_gb_tuned.predict(X_test_raw))

Wir kommen leider nicht unter die 7.4% vorerst

---
###

## Ohne Zinssatz

In [ ]:
spalten_weg = ['loan_int_rate']
# falls ihr die Indikatorspalte eingebaut habt, die auch raus:
# spalten_weg = ['loan_int_rate', 'int_rate_fehlt']

X_train_ohne = X_train_raw.drop(columns=spalten_weg)
X_test_ohne = X_test_raw.drop(columns=spalten_weg)

model_ohne_zins = HistGradientBoostingClassifier(**suche.best_params_, random_state=42)
model_ohne_zins.fit(X_train_ohne, y_train)

bewerten('Gradient Boosting (getunt, mit Zins)', y_test, model_gb_tuned.predict(X_test_raw))
bewerten('Gradient Boosting (getunt, ohne Zins)', y_test, model_ohne_zins.predict(X_test_ohne))

## ACHTUNG. Vielleicht ist es Overfitted. Man weiß nicht genau.

---
## Wie performt das auf Zinssätze trainierte Modell wenn die 'interest_rate' NaN ist?

Wir können beweisen, dass das Modell 'Zins_Modell' overfitted ist.
Die loan_interest ist eine transitiver Wert, der sich aus dem Risiko im Nachhinein ergibt und somit für die Bewertung eigentlich noch nicht zur Verfügung steht aber eben stark korreliert


In [ ]:
#Datensatz ohne Zins
X_test_nan = X_test_raw.copy()
X_test_nan['loan_int_rate'] = np.nan

bewerten('Zins_Modell angewendet auf vollen Testdatensatz', y_test, model_gb_tuned.predict(X_test_raw))
bewerten('Ohne_Zins_Modell angewendet auf Datensatz ohne Zins', y_test, model_ohne_zins.predict(X_test_ohne))
bewerten('Zins_Modell angewendet auf NaN', y_test, model_gb_tuned.predict(X_test_nan))